# Modulo 2 - Clasificacion de conduccion distractiva\n\nNotebook de entrenamiento en Google Colab para el dataset Multi-Class Driver Behavior Image Dataset.

In [ ]:
!pip install -q torch torchvision pandas scikit-learn matplotlib seaborn pillow kaggle\n\nfrom pathlib import Path\nimport json\nimport os\nimport shutil\nimport sys\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport seaborn as sns\nimport torch\n\nPROJECT_ROOT = Path('/content/rna-sistema_de_transporte_inteligente')\nif PROJECT_ROOT.exists():\n    os.chdir(PROJECT_ROOT)\nelse:\n    print('Clona o sube el repositorio en /content/rna-sistema_de_transporte_inteligente antes de continuar.')\nsys.path.insert(0, str(Path.cwd()))\n\ntorch.manual_seed(42)\ndevice = 'cuda' if torch.cuda.is_available() else 'cpu'\ndevice

## Credenciales Kaggle\n\nSube `kaggle.json` al entorno de Colab o define `KAGGLE_USERNAME` y `KAGGLE_KEY` como variables de entorno.

In [ ]:
kaggle_dir = Path.home() / '.kaggle'\nkaggle_dir.mkdir(exist_ok=True)\nuploaded_json = Path('/content/kaggle.json')\nif uploaded_json.exists():\n    shutil.copy(uploaded_json, kaggle_dir / 'kaggle.json')\n    os.chmod(kaggle_dir / 'kaggle.json', 0o600)\n\nDATA_ROOT = Path('data/raw/multi_class_driver_behavior')\nDATA_ROOT.mkdir(parents=True, exist_ok=True)\n!python scripts/download_data.py --output-dir data/raw/multi_class_driver_behavior

In [ ]:
from src.config import DEFAULT_BATCH_SIZE, DEFAULT_IMAGE_SIZE, DRIVER_BEHAVIOR_CLASSES\nfrom src.module2_distraction.data_loader import create_dataloaders\nfrom src.module2_distraction.model import build_driver_behavior_model\nfrom src.module2_distraction.trainer import TrainingConfig, fit\n\nloaders, splits = create_dataloaders(\n    DATA_ROOT,\n    image_size=DEFAULT_IMAGE_SIZE,\n    batch_size=DEFAULT_BATCH_SIZE,\n    num_workers=2,\n    val_ratio=0.15,\n    test_ratio=0.15,\n    seed=42,\n    strict_classes=False,\n)\nclass_names = [splits.idx_to_class[i] for i in range(len(splits.idx_to_class))]\nprint(class_names)\nprint({name: len(dataset) for name, dataset in [('train', splits.train), ('val', splits.val), ('test', splits.test)]})

In [ ]:
model = build_driver_behavior_model(\n    num_classes=len(class_names),\n    model_name='resnet18',\n    pretrained=True,\n    freeze_backbone=False,\n)\n\nconfig = TrainingConfig(\n    epochs=12,\n    learning_rate=1e-4,\n    weight_decay=1e-4,\n    patience=4,\n    device=device,\n    checkpoint_path='models/distraction/best_model.pt',\n    model_name='resnet18',\n    image_size=DEFAULT_IMAGE_SIZE,\n    extra_metadata={'dataset': 'arafatsahinafridi/multi-class-driver-behavior-image-dataset'},\n)\n\ntraining_summary = fit(model, loaders, class_names, splits.class_to_idx, config)\ntraining_summary

In [ ]:
from src.module2_distraction.evaluator import collect_predictions, evaluate_predictions, save_evaluation_artifacts, distraction_frequency_report\n\nartifact_dir = Path('models/distraction/artifacts')\nartifact_dir.mkdir(parents=True, exist_ok=True)\ny_true, y_pred, probabilities = collect_predictions(model, loaders['test'], device)\nresults = evaluate_predictions(y_true, y_pred, class_names)\nresults['distraction_frequency'] = distraction_frequency_report(y_pred, class_names)\nsave_evaluation_artifacts(results, artifact_dir)\npd.DataFrame(results['classification_report'])

In [ ]:
cm = np.array(results['confusion_matrix'])\nplt.figure(figsize=(8, 6))\nsns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)\nplt.xlabel('Prediccion')\nplt.ylabel('Etiqueta real')\nplt.tight_layout()\nplt.savefig(artifact_dir / 'confusion_matrix.png', dpi=160)\nplt.show()

In [ ]:
from src.module2_distraction.augmentation import IMAGENET_MEAN, IMAGENET_STD\n\ndef denormalize(tensor):\n    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)\n    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)\n    image = tensor.cpu() * std + mean\n    return image.clamp(0, 1).permute(1, 2, 0).numpy()\n\ndef save_prediction_grid(indices, filename, title):\n    if len(indices) == 0:\n        print(f'Sin ejemplos para {title}')\n        return\n    indices = indices[:9]\n    fig, axes = plt.subplots(3, 3, figsize=(10, 10))\n    axes = axes.flatten()\n    test_dataset = splits.test.dataset\n    for ax, pred_index in zip(axes, indices):\n        dataset_index = splits.test.indices[pred_index]\n        image, _ = test_dataset[dataset_index]\n        ax.imshow(denormalize(image))\n        true_name = class_names[y_true[pred_index]]\n        pred_name = class_names[y_pred[pred_index]]\n        confidence = probabilities[pred_index].max()\n        ax.set_title(f'Real: {true_name}\nPred: {pred_name} ({confidence:.2f})')\n        ax.axis('off')\n    for ax in axes[len(indices):]:\n        ax.axis('off')\n    fig.suptitle(title)\n    plt.tight_layout()\n    plt.savefig(artifact_dir / filename, dpi=160)\n    plt.show()\n\ncorrect = np.where(y_true == y_pred)[0].tolist()\nerrors = np.where(y_true != y_pred)[0].tolist()\nsave_prediction_grid(correct, 'correct_predictions.png', 'Ejemplos correctamente clasificados')\nsave_prediction_grid(errors, 'error_cases.png', 'Casos erroneos')

In [ ]:
pd.DataFrame(results['distraction_frequency']).to_csv(artifact_dir / 'distraction_frequency.csv', index=False)\npd.DataFrame(results['distraction_frequency'])